In [ ]:


# PROJECT 14: AIRLINE FLIGHT DELAY ANALYSIS

import pandas as pd
import numpy as np

 # 1. LOAD DATASET
df = pd.read_csv("14_flight_delays.csv")

print("First five rows:")
print(df.head())

print("\nDataset Shape:")
print(df.shape)

print("\nColumn Names:")
print(df.columns)

print("\nData Types:")
print(df.dtypes)


First five rows:
  Flight_ID Flight_Date    Airline Origin Destination Departure_Time  \
0    F00000  2026-08-23  Air India    MAA         MAA        Evening   
1    F00001  2026-08-04     IndiGo    HYD         BOM      Afternoon   
2    F00002  2026-10-31    Vistara    BOM         MAA      Afternoon   
3    F00003  2025-11-22   SpiceJet    CCU         MAA      Afternoon   
4    F00004  2025-08-29   SpiceJet    HYD         HYD          Night   

   Departure_Delay_Mins  Arrival_Delay_Mins Delay_Cause Canceled Diverted  
0                  47.1                 0.0     Carrier       No       No  
1                   0.0                35.3     Weather       No       No  
2                  38.0                 1.7     Weather       No       No  
3                  35.0                 9.1     Weather       No       No  
4                  12.4                 6.3         NaN       No       No  

Dataset Shape:
(913, 11)

Column Names:
Index(['Flight_ID', 'Flight_Date', 'Airline', 'Origin

In [4]:
# 2. CHECK MISSING VALUES AND DUPLICATES
print("Missing Values:")
print(df.isnull().sum())

print("\nDuplicate Flight IDs:")
print(df["Flight_ID"].duplicated().sum())

# Remove exact duplicate flight records
df = df.drop_duplicates(subset="Flight_ID", keep="first").copy()

print("\nShape after removing duplicates:")
print(df.shape)

Missing Values:
Flight_ID                 0
Flight_Date               0
Airline                   0
Origin                    0
Destination               0
Departure_Time            0
Departure_Delay_Mins      0
Arrival_Delay_Mins       22
Delay_Cause             193
Canceled                  0
Diverted                  0
dtype: int64

Duplicate Flight IDs:
13

Shape after removing duplicates:
(900, 11)


In [5]:
# 3. HANDLE MISSING ARRIVAL DELAYS
median_arrival = df["Arrival_Delay_Mins"].median()

df["Arrival_Delay_Mins"] = df["Arrival_Delay_Mins"].fillna(
    median_arrival
)

print("\nMissing Arrival Delays After Cleaning:")
print(df["Arrival_Delay_Mins"].isnull().sum())

print("Median used:", median_arrival)



Missing Arrival Delays After Cleaning:
0
Median used: 24.75


In [6]:
# 4. CONVERT DATE AND EXTRACT MONTH AND DAY
df["Flight_Date"] = pd.to_datetime(df["Flight_Date"])

df["Month"] = df["Flight_Date"].dt.month

df["Day_of_Week"] = df["Flight_Date"].dt.day_name()

print("\nDate Information:")
print(df[["Flight_Date", "Month", "Day_of_Week"]].head())


Date Information:
  Flight_Date  Month Day_of_Week
0  2026-08-23      8      Sunday
1  2026-08-04      8     Tuesday
2  2026-10-31     10    Saturday
3  2025-11-22     11    Saturday
4  2025-08-29      8      Friday


In [7]:
# 5. TOTAL FLIGHTS, AVERAGE DELAY AND ON-TIME %
total_flights = len(df)

average_delay = df["Departure_Delay_Mins"].mean()

on_time_flights = (
    df["Departure_Delay_Mins"] <= 15
).sum()

on_time_percentage = (
    on_time_flights / total_flights
) * 100

print("\nFLIGHT PERFORMANCE")
print("Total Flights:", total_flights)

print("Average Departure Delay:", round(average_delay, 2))

print("On-Time Flights:", on_time_flights)

print("On-Time Percentage:", round(on_time_percentage, 2), "%")


FLIGHT PERFORMANCE
Total Flights: 900
Average Departure Delay: 28.73
On-Time Flights: 365
On-Time Percentage: 40.56 %


In [8]:
# 6. RANK ORIGIN AIRPORTS BY AVERAGE DEPARTURE DELAY
airport_ranking = df.groupby("Origin").agg(
    Total_Flights=("Flight_ID", "count"),
    Average_Delay=("Departure_Delay_Mins", "mean")
)

airport_ranking = airport_ranking.sort_values(
    by="Average_Delay", ascending=False
)

print("\nAIRPORT DELAY RANKING")
print(airport_ranking.round(2))


AIRPORT DELAY RANKING
        Total_Flights  Average_Delay
Origin                              
BOM               156          31.58
DEL               154          30.58
BLR               150          30.32
HYD               149          29.05
CCU               148          27.46
MAA               143          22.94


In [9]:
# 7. TOP 10 ROUTES WITH HIGHEST ARRIVAL DELAYS
df["Route"] = df["Origin"] + " to " + df["Destination"]

route_delays = df.groupby("Route").agg(
    Total_Flights=("Flight_ID", "count"),
    Average_Arrival_Delay=("Arrival_Delay_Mins", "mean")
)

top_10_routes = route_delays.sort_values(
    by="Average_Arrival_Delay",
    ascending=False
).head(10)

print("\nTOP 10 WORST ROUTES")
print(top_10_routes.round(2))


TOP 10 WORST ROUTES
            Total_Flights  Average_Arrival_Delay
Route                                           
DEL to CCU             22                  44.33
MAA to MAA             25                  40.07
DEL to DEL             21                  40.04
HYD to CCU             21                  38.73
HYD to HYD             21                  38.40
CCU to BLR             25                  38.25
BLR to CCU             18                  37.15
BLR to BOM             30                  36.67
HYD to MAA             24                  36.24
MAA to HYD             32                  35.31


In [10]:
# 8. COMPARE DELAYS ACROSS TIMES OF DAY
# Departure_Time already contains Morning, Afternoon,
# Evening and Night categories in this dataset.

df["Is_Delayed"] = df["Departure_Delay_Mins"] > 15

time_analysis = df.groupby("Departure_Time").agg(
    Total_Flights=("Flight_ID", "count"),
    Average_Delay=("Departure_Delay_Mins", "mean"),
    Delay_Rate=("Is_Delayed", "mean")
)

time_analysis["Delay_Rate"] *= 100

time_analysis = time_analysis.reindex(
    ["Morning", "Afternoon", "Evening", "Night"]
)

print("\nTIME OF DAY ANALYSIS")
print(time_analysis.round(2))



TIME OF DAY ANALYSIS
                Total_Flights  Average_Delay  Delay_Rate
Departure_Time                                          
Morning                   177          28.62       59.89
Afternoon                 252          30.16       60.32
Evening                   230          28.57       60.87
Night                     241          27.46       56.85


In [11]:
# 9. DELAY CAUSE BREAKDOWN
cause_analysis = df["Delay_Cause"].fillna(
    "Not recorded"
).value_counts()

print("\nDELAY CAUSES")
print(cause_analysis)

print("\nDelay Cause Percentages:")
print(
    (cause_analysis / total_flights * 100).round(2)
)


DELAY CAUSES
Delay_Cause
Not recorded    191
Weather         180
Carrier         178
Security        177
Air Traffic     174
Name: count, dtype: int64

Delay Cause Percentages:
Delay_Cause
Not recorded    21.22
Weather         20.00
Carrier         19.78
Security        19.67
Air Traffic     19.33
Name: count, dtype: float64


In [12]:
# 10. CANCELLATION AND DIVERSION PERCENTAGES
airline_analysis = df.groupby("Airline").agg(
    Total_Flights=("Flight_ID", "count"),
    Cancellations=("Canceled", lambda x: (x == "Yes").mean() * 100),
    Diversions=("Diverted", lambda x: (x == "Yes").mean() * 100)
)

print("\nAIRLINE CANCELLATION AND DIVERSION ANALYSIS")
print(airline_analysis.round(2))


AIRLINE CANCELLATION AND DIVERSION ANALYSIS
           Total_Flights  Cancellations  Diversions
Airline                                            
Air India            170           3.53        2.35
Akasa                172           5.23        1.74
IndiGo               180           3.33        2.78
SpiceJet             194           2.58        0.00
Vistara              184           4.89        2.17


In [13]:
# 11. CORRELATION BETWEEN DEPARTURE AND ARRIVAL DELAYS
# Use original non-missing arrival values for correlation.

raw_arrival = pd.read_csv("14_flight_delays.csv")
raw_arrival = raw_arrival.drop_duplicates(
    subset="Flight_ID", keep="first"
)

correlation = raw_arrival[
    "Departure_Delay_Mins"
].corr(
    raw_arrival["Arrival_Delay_Mins"]
)

print("\nCORRELATION:")
print(round(correlation, 3))


CORRELATION:
0.068


In [14]:
# 12. NUMPY STATISTICS
delays = df["Departure_Delay_Mins"].to_numpy()

median_delay = np.median(delays)

standard_deviation = np.std(delays)

percentile_90 = np.percentile(delays, 90)

print("\nNUMPY STATISTICS")
print("Median:", round(median_delay, 2))

print("Standard Deviation:", round(standard_deviation, 2))

print("90th Percentile:", round(percentile_90, 2))




NUMPY STATISTICS
Median: 23.6
Standard Deviation: 27.76
90th Percentile: 70.8


In [15]:
# 13. IDENTIFY OUTLIERS USING IQR
Q1 = np.percentile(delays, 25)

Q3 = np.percentile(delays, 75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR

upper_limit = Q3 + 1.5 * IQR

outliers = df[
    (df["Departure_Delay_Mins"] < lower_limit) |
    (df["Departure_Delay_Mins"] > upper_limit)
]

print("\nOUTLIERS")
print("Q1:", Q1)

print("Q3:", Q3)

print("IQR:", IQR)

print("Lower Limit:", lower_limit)

print("Upper Limit:", upper_limit)

print("Outlier Count:", len(outliers))

print(outliers[["Flight_ID", "Departure_Delay_Mins"]])



OUTLIERS
Q1: 0.0
Q3: 46.2
IQR: 46.2
Lower Limit: -69.30000000000001
Upper Limit: 115.50000000000001
Outlier Count: 2
    Flight_ID  Departure_Delay_Mins
425    F00425                 122.4
470    F00470                 116.1


In [16]:
# 14. CREATE DELAY SEVERITY CATEGORY

def classify_delay(delay):

    if delay <= 15:
        return "On Time"

    elif delay <= 60:
        return "Minor Delay"

    elif delay <= 180:
        return "Moderate Delay"

    else:
        return "Severe Delay"


df["Delay_Severity"] = df["Departure_Delay_Mins"].apply(
    classify_delay
)

print("\nDELAY SEVERITY")
print(df["Delay_Severity"].value_counts())


DELAY SEVERITY
Delay_Severity
Minor Delay       398
On Time           365
Moderate Delay    137
Name: count, dtype: int64


In [17]:
# 15. OPERATIONAL RECOMMENDATIONS

print("\nOPERATIONAL RECOMMENDATIONS")

print("1. Improve airport ground handling.")

print("2. Monitor weather conditions.")

print("3. Improve airline scheduling.")

print("4. Coordinate better with air traffic control.")

print("5. Monitor high-delay routes and airports.")

print("\nPROJECT COMPLETED SUCCESSFULLY!")



OPERATIONAL RECOMMENDATIONS
1. Improve airport ground handling.
2. Monitor weather conditions.
3. Improve airline scheduling.
4. Coordinate better with air traffic control.
5. Monitor high-delay routes and airports.

PROJECT COMPLETED SUCCESSFULLY!
